In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import os

drive.mount('/content/drive')
folder_path = '/content/drive/My Drive/AIHACKATHON/data'
file_path_attrition = os.path.join(folder_path, 'Attrition.csv')
df = pd.read_csv(file_path_attrition)
df.head()

Mounted at /content/drive


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [2]:
# ==========================================
# 第一部分：业务逻辑过滤 (Filtering)
# ==========================================
# 核心逻辑：只关注高绩效员工 (Key Members)，剔除低绩效噪音
df = df[df['PerformanceRating'] >= 3].copy()

# 剔除逻辑错误数据 (总工龄必须大于等于司龄)
df = df[df['TotalWorkingYears'] >= df['YearsAtCompany']]


# 2026 西雅图市场月薪中位数映射 (单位：USD)
# 考虑了西雅图的高生活成本和 2025-2026 的 AI 岗位溢价
seattle_2026_market_map = {
    'Research Scientist': 15500,     # AI/Deep Learning 方向涨幅极大
    'Manager': 19000,                # 资深管理岗，包含大量 Stock
    'Manufacturing Director': 16500,  # 制造业在西雅图（如波音或自动化工厂）的水平
    'Research Director': 24000,      # 高级科研领导者
    'Sales Executive': 12500,        # 含高额佣金提成
    'Healthcare Representative': 9500,# 医疗保健行业相对稳定
    'Human Resources': 9200,         # 西雅图大厂 HR 薪资较高
    'Laboratory Technician': 7000,   # 基础技术岗
    'Sales Representative': 5500     # 初级销售
}

# 1. 模拟数据对齐：将 IBM 老数据的 MonthlyIncome 缩放到 2026 水平 (假设 1.7 倍增长)
df['MonthlyIncome_2026'] = df['MonthlyIncome'] * 1.7

# 2. 映射市场基准
df['Market_Median_2026'] = df['JobRole'].map(seattle_2026_market_map)

# 3. 注入内部相对排名 (非常重要，体现审计逻辑) ---
df['Internal_Salary_Rank'] = df.groupby(['JobRole'])['MonthlyIncome'].rank(pct=True)

# 4. 模拟绩效稳定性 (为 Key Member 量身定制) ---
# 既然是 Key Member (Rating >= 3)，我们假设他们过去也表现优秀
df['Performance_Consistency'] = np.random.uniform(0.8, 1.0, len(df))

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,MonthlyIncome_2026,Market_Median_2026,Internal_Salary_Rank,Performance_Consistency
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,0,1,6,4,0,5,10188.1,12500,0.460123,0.885528
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,3,3,10,7,1,7,8721.0,15500,0.924658,0.891368
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,3,3,0,0,0,0,3553.0,7000,0.092664,0.815371
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,3,8,7,3,0,4945.3,15500,0.513699,0.961264
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,3,3,2,2,2,2,5895.6,7000,0.637066,0.921404


In [3]:
# ==========================================
# 第二部分：Outlier 处理与数据准备
# ==========================================

# 对薪水进行对数平滑，处理极端高薪对模型的影响
df['Income_Log'] = np.log1p(df['MonthlyIncome_2026'])


df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,MonthlyIncome_2026,Market_Median_2026,Internal_Salary_Rank,Performance_Consistency,Income_Log
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,6,4,0,5,10188.1,12500,0.460123,0.885528,9.229074
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,3,10,7,1,7,8721.0,15500,0.924658,0.891368,9.073604
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,3,0,0,0,0,3553.0,7000,0.092664,0.815371,8.175829
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,8,7,3,0,4945.3,15500,0.513699,0.961264,8.506395
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,3,2,2,2,2,5895.6,7000,0.637066,0.921404,8.682131


In [6]:
# ==========================================
# 第三部分：Nan 处理
# ==========================================

df.isnull().sum()

,0
Age,0
Attrition,0
BusinessTravel,0
DailyRate,0
Department,0
DistanceFromHome,0
Education,0
EducationField,0
EmployeeCount,0
EmployeeNumber,0


In [7]:
# --- 定义 Agent 2 专属特征池 ---
# 剔除掉所有“结果类”和“干扰类”字段
drop_list = [
    'Income_Log', 'MonthlyIncome_2026', 'MonthlyIncome', # 目标变量及其原值
    'Attrition', 'Target',                               # 离职标签 (留给 Agent 3)
    'Compa_Ratio', 'Adjustment_Needed_Pct',              # 衍生计算项 (审计产出，非输入)
    'Initial_Compa_Ratio',                               # 泄露项
    'EmployeeCount', 'EmployeeNumber', 'StandardHours',  # 无意义常量
    'JobSatisfaction', 'EnvironmentSatisfaction',        # 软性情绪 (留给 Agent 4)
    'RelationshipSatisfaction', 'WorkLifeBalance',       # 软性情绪 (留给 Agent 4)
    'YearsSinceLastPromotion', 'OverTime',               # 压力因子 (留给 Agent 3)
    'OverTime_Numeric', 'Age', 'Gender'
]

# 最终确定的 X 和 y
X = df.drop(columns=[c for c in drop_list if c in df.columns])
y = df['Income_Log']

print(f"y 形状: {y.shape}")
print(f"X 形状: {X.shape}")
print(f"参与训练的特征: {X.columns.tolist()}")

y 形状: (1470,)
X 形状: (1470, 25)
参与训练的特征: ['BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'MaritalStatus', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'PercentSalaryHike', 'PerformanceRating', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsWithCurrManager', 'Market_Median_2026', 'Internal_Salary_Rank', 'Performance_Consistency']


In [8]:
from sklearn.model_selection import train_test_split

# 1. 第一步：分出 80% 训练集，剩下 20% 做验证+测试
# 使用 stratify=X['JobLevel'] 确保职级分布均衡
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=X['JobLevel'] # 这里的含金量在于保证了定薪维度的公平
)

# 2. 第二步：把剩下的 20% 对半分 (10% Val, 10% Test)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

print(f"训练集规模: {X_train.shape[0]} (学习身价规律)")
print(f"验证集规模: {X_val.shape[0]} (防止训练过头)")
print(f"测试集规模: {X_test.shape[0]} (模拟盲测审计)")

训练集规模: 1176 (学习身价规律)
验证集规模: 147 (防止训练过头)
测试集规模: 147 (模拟盲测审计)


In [13]:
import lightgbm as lgb
import joblib
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

# 1. 自动识别分类特征 (JobRole, Department 等)
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# 2. 配置 LightGBM 参数
# 使用 objective='huber' 来降低极高薪对模型的影响
params = {
    'objective': 'huber',        # 对离群值鲁棒
    'metric': 'mae',             # 使用平均绝对误差监控
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.02,       # 较低的学习率配合更多的迭代
    'num_leaves': 31,
    'max_depth': 6,              # 限制深度防止过拟合
    'feature_fraction': 0.8,     # 每次迭代随机选80%特征
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 0.1,            # L1 正则化
    'lambda_l2': 0.1,            # L2 正则化
    'random_state': 42
}

# 3. 训练模型 (带 Early Stopping)
print("开始训练 Agent 2 薪酬审计模型...")
model_a2 = lgb.LGBMRegressor(**params, n_estimators=2000)

model_a2.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100), # 100轮不降则停
        lgb.log_evaluation(period=50)            # 每50轮打印一次
    ]
)

# 4. 在测试集上评估审计精度 (还原真实金额)
preds_log = model_a2.predict(X_test)
y_test_real = np.expm1(y_test)
preds_real = np.expm1(preds_log)

mae_usd = mean_absolute_error(y_test_real, preds_real)
r2 = r2_score(y_test, preds_log)

print(f"\n审计模型训练完成！")
print(f"测试集 R² (对数空间): {r2:.4f}")
print(f"平均审计误差 (MAE): ${mae_usd:.2f} (USD)")

# 5. 导出模型文件 (供后续 Agent 调用)
model_filename = '/content/drive/My Drive/AIHACKATHON/model/agent2_salary_regressor.pkl'
joblib.dump(model_a2, model_filename)
print(f"模型已导出为: {model_filename}")

开始训练 Agent 2 薪酬审计模型...
Training until validation scores don't improve for 100 rounds
[50]	valid_0's l1: 0.213995
[100]	valid_0's l1: 0.0924388
[150]	valid_0's l1: 0.048292
[200]	valid_0's l1: 0.0342073
[250]	valid_0's l1: 0.0295838
[300]	valid_0's l1: 0.0276325
[350]	valid_0's l1: 0.0272291
[400]	valid_0's l1: 0.0270795
[450]	valid_0's l1: 0.0266589
[500]	valid_0's l1: 0.026482
[550]	valid_0's l1: 0.0265686
Early stopping, best iteration is:
[480]	valid_0's l1: 0.0264482

审计模型训练完成！
测试集 R² (对数空间): 0.9949
平均审计误差 (MAE): $317.16 (USD)
模型已导出为: /content/drive/My Drive/AIHACKATHON/model/agent2_salary_regressor.pkl


In [14]:
import joblib
import pandas as pd
import numpy as np

# 1. 加载你刚刚保存的模型
model_path = '/content/drive/My Drive/AIHACKATHON/model/agent2_salary_regressor.pkl'
loaded_model = joblib.load(model_path)

# 1. 获取训练模型时用到的特征列表
# 假设你的模型对象是 loaded_model
model_features = loaded_model.feature_name_

# 2. 从你的总表 df 中取一行作为模板，确保所有列都在
# 这样能保证  BusinessTravel 等所有训练时用到的列一个不落
sample_row = df[model_features].iloc[0:1].copy()

# 3. 把你想测试的数据填进去 (覆盖掉 sample_row)
sample_row['Market_Median_2026'] = 15500
sample_row['JobLevel'] = 4
sample_row['JobRole'] = 'Research Scientist'
sample_row['TotalWorkingYears'] = 10
sample_row['PerformanceRating'] = 4
sample_row['Internal_Salary_Rank'] = 0.95
# ... 你可以根据需要修改其他字段，没改的会保持默认值

# 4. 统一转换类型 (非常重要)
for col in sample_row.columns:
    if df[col].dtype.name == 'category' or df[col].dtype == 'object':
        sample_row[col] = sample_row[col].astype('category')

# 5. 再次执行预测
pred_log = loaded_model.predict(sample_row)
predicted_salary = np.expm1(pred_log)[0]

print(f"Agent 2 评估身价: ${predicted_salary:.2f}")

Agent 2 评估身价: $15564.76
